In [1]:
import glob
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

t_start = time.time()

# 1. Load data
input_files = glob.glob("/kaggle/input/**/*", recursive=True)
train_path = next((p for p in input_files if "train" in os.path.basename(p)), "train.csv.gz")
test_path  = next((p for p in input_files if "test" in os.path.basename(p)), "test.csv")
sub_path   = next((p for p in input_files if "sample_submission" in os.path.basename(p)), "sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
sub_df   = pd.read_csv(sub_path)

TOP_FACILITIES = [
    "FCL-007", "FCL-012", "FCL-035", "FCL-055", "FCL-056", "FCL-058",
    "FCL-072", "FCL-077", "FCL-078", "FCL-080", "FCL-085", "FCL-087",
    "FCL-093", "FCL-104", "FCL-135", "FCL-136", "FCL-137", "FCL-138", "FCL-139", "FCL-141"
]

# 2. Fast vectorized feature engineering
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "facilities" in out.columns:
        s = out["facilities"].fillna("").astype(str)
        out["facility_count"] = np.where(s.str.len() > 2, s.str.count(",") + 1, 0).astype("float32")
        for code in TOP_FACILITIES:
            out[f"has_{code}"] = s.str.contains(code, regex=False).astype(np.int8)

    for dcol in ["snapshot_start_date", "snapshot_end_date", "first_review_date", "last_review_date"]:
        if dcol in out.columns:
            dt = pd.to_datetime(out[dcol], errors="coerce")
            out[f"{dcol}_yr"] = dt.dt.year.astype("float32")
            out[f"{dcol}_mo"] = dt.dt.month.astype("float32")

    if "review_count" in out.columns:
        out["has_reviews"] = (out["review_count"] > 0).astype(np.int8)
    if "borough" in out.columns:
        out["has_borough"] = out["borough"].notna().astype(np.int8)
    out["missing_cnt"] = out.isna().sum(axis=1).astype("float32")

    if {"max_seats", "workstations"}.issubset(out.columns):
        out["seats_per_ws"] = (out["max_seats"] / (out["workstations"].fillna(0) + 1.0)).astype("float32")
    if {"operator_metro_units", "operator_portfolio_size"}.issubset(out.columns):
        out["metro_share"] = (out["operator_metro_units"] / (out["operator_portfolio_size"].fillna(0) + 1.0)).astype("float32")
    if {"booked_load", "max_seats"}.issubset(out.columns):
        out["load_per_seat"] = (out["booked_load"] / (out["max_seats"] + 1.0)).astype("float32")

    for col, cap in [("max_term_days", 3650.0), ("max_term_floor", 3650.0),
                      ("max_term_ceiling", 3650.0), ("max_term_avg", 3650.0),
                      ("reviews_per_month", 30.0)]:
        if col in out.columns:
            out[col] = np.clip(out[col].astype("float32"), 0.0, cap)

    drop_cols = [c for c in ["facilities", "snapshot_start_date", "snapshot_end_date",
                             "first_review_date", "last_review_date", "price_tier"] if c in out.columns]
    return out.drop(columns=drop_cols)

y = train_df["price_tier"].astype(np.int8)
X = build_features(train_df)
X_test = build_features(test_df)

# Frequency features for high-cardinality columns (fast & compact)
for hc in ["operator_ref", "zone", "space_class"]:
    if hc in X.columns:
        freq = X[hc].value_counts(normalize=True)
        X[f"{hc}_freq"] = X[hc].map(freq).fillna(0.0).astype("float32")
        X_test[f"{hc}_freq"] = X_test[hc].map(freq).fillna(0.0).astype("float32")

X = X.drop(columns=["operator_ref"], errors="ignore")
X_test = X_test.drop(columns=["operator_ref"], errors="ignore")

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# 3. Fast preprocessing pipeline
prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), cat_cols),
])

# 4. Compare 3 distinct model families with 3-fold Stratified CV
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

models = {
    "LogisticRegression": Pipeline([
        ("prep", prep),
        ("clf", LogisticRegression(max_iter=400, tol=1e-3, random_state=42)),
    ]),
    "RandomForest": Pipeline([
        ("prep", prep),
        ("clf", RandomForestClassifier(
            n_estimators=100, max_depth=15, min_samples_leaf=10,
            n_jobs=-1, random_state=42
        )),
    ]),
    "HistGradientBoosting": Pipeline([
        ("prep", prep),
        ("clf", HistGradientBoostingClassifier(
            learning_rate=0.08, max_iter=150, max_leaf_nodes=63,
            min_samples_leaf=30, random_state=42
        )),
    ]),
}

scores_map = {}
for name, pipe in models.items():
    t_m = time.time()
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy", n_jobs=1)
    scores_map[name] = scores.mean()
    print(f"{name:22s} | CV Acc: {scores.mean():.5f} (+/- {scores.std():.5f}) | {time.time() - t_m:.1f}s")

# 5. Fit best model on full train and write submission.csv
best_name = max(scores_map, key=scores_map.get)
print(f"\nBest model: {best_name} ({scores_map[best_name]:.5f})")

best_pipe = models[best_name]
best_pipe.fit(X, y)
preds = best_pipe.predict(X_test)

sub = pd.DataFrame({
    "ID": sub_df["ID"],
    "price_tier": preds.astype(int),
})
sub.to_csv("submission.csv", index=False)
print(f"Done in {time.time() - t_start:.1f}s | submission shape: {sub.shape}")

LogisticRegression     | CV Acc: 0.52737 (+/- 0.00185) | 196.9s
RandomForest           | CV Acc: 0.68372 (+/- 0.00144) | 87.4s
HistGradientBoosting   | CV Acc: 0.71307 (+/- 0.00121) | 210.7s

Best model: HistGradientBoosting (0.71307)
Done in 610.9s | submission shape: (114538, 2)
